# TP3 – Détection de fraude bancaire avec un MLP
**Module : Deep Learning & Reinforcement Learning**  
**Filière : IITE-2 | Pr. I. HAMZAOUI | Année 2025/2026**

In [2]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Pour reproduire les résultats
np.random.seed(42)
tf.random.set_seed(42)

## Q1) Chargement des données

In [4]:
# Chargement du dataset
df = pd.read_csv('dataset_fraude.csv')

# Dimensions du dataset
print("Dimensions du dataset :", df.shape)
print(f"  → {df.shape[0]} transactions ou les lignes du dataset , {df.shape[1]} variables  sont les colonnes du dataset amount ... \n")

# visualisation du Premières lignes
print("Premières lignes du dataset :")
df.head()

Dimensions du dataset : (5228, 11)
  → 5228 transactions ou les lignes du dataset , 11 variables  sont les colonnes du dataset amount ... 

Premières lignes du dataset :


,amount,hour,merchant_category,country,country_risk,distance_km,tx_last_1h,tx_last_24h,device_change,is_international,label_fraud
0,11.95,12,online_services,MA,0.044,13.91,4,8,0,0,1
1,10.29,11,utilities,MA,0.136,13.17,1,4,0,0,0
2,97.04,19,pharmacy,MA,0.075,51.35,0,2,0,0,0
3,16.60,19,atm,MA,0.047,21.52,3,7,0,0,0
4,70.93,7,fuel,CN,0.396,47.09,0,2,0,1,0


## Q2) Analyse de la variable cible

In [7]:
# Avec value_counts() de Pandas
print("=== Avec value_counts() ===")
counts = df['label_fraud'].value_counts()
print(counts)
print(f"\nTransactions normales  (0) : {counts[0]}")
print(f"Transactions frauduleuses (1) : {counts[1]}")
print(f"Taux de fraude : {counts[1] / len(df) * 100:.2f}%")

# Avec bincount de Numpy
print("\n=== Avec np.bincount() ===")
bc = np.bincount(df['label_fraud'])
print(f"Normal (0) : {bc[0]}")
print(f"Fraude (1) : {bc[1]}")


=== Avec value_counts() ===
label_fraud
0    4451
1     777
Name: count, dtype: int64

Transactions normales  (0) : 4451
Transactions frauduleuses (1) : 777
Taux de fraude : 14.86%

=== Avec np.bincount() ===
Normal (0) : 4451
Fraude (1) : 777


**Analyse Q2 :** Le dataset présente un déséquilibre de classes notable. Les transactions frauduleuses représentent une minorité des observations. Ce déséquilibre devra être pris en compte lors de l'entraînement du modèle.

## Q3) Séparation des variables

In [8]:
# Séparation variables explicatives (X) et variable cible (y)
X = df.drop('label_fraud', axis=1)
y = df['label_fraud']

print("Dimensions de X (variables explicatives) :", X.shape)
print("Dimensions de y (variable cible) :", y.shape)
print("\nVariables explicatives :")
print(X.columns.tolist())

Dimensions de X (variables explicatives) : (5228, 10)
Dimensions de y (variable cible) : (5228,)

Variables explicatives :
['amount', 'hour', 'merchant_category', 'country', 'country_risk', 'distance_km', 'tx_last_1h', 'tx_last_24h', 'device_change', 'is_international']


## Q4) Encodage des variables catégorielles

In [9]:
# Vérification des types avant encodage
print("Types des variables avant encodage :")
print(X.dtypes)
print("\nVariables catégorielles à encoder : ['merchant_category', 'country']")

# Encodage One-Hot des variables catégorielles avec pd.get_dummies
X_encoded = pd.get_dummies(X, columns=['merchant_category', 'country'])#on a pas utilise drop_first comme vous avez dit madam

print("\nDimensions après encodage :", X_encoded.shape)
print(f"  → Avant : {X.shape[1]} variables | Après : {X_encoded.shape[1]} variables")
print("\nNouvelles colonnes (extrait) :")
print([c for c in X_encoded.columns if 'merchant' in c or 'country' in c][:15])

Types des variables avant encodage :
amount               float64
hour                   int64
merchant_category     object
country               object
country_risk         float64
distance_km          float64
tx_last_1h             int64
tx_last_24h            int64
device_change          int64
is_international       int64
dtype: object

Variables catégorielles à encoder : ['merchant_category', 'country']

Dimensions après encodage : (5228, 30)
  → Avant : 10 variables | Après : 30 variables

Nouvelles colonnes (extrait) :
['country_risk', 'merchant_category_atm', 'merchant_category_electronics', 'merchant_category_fashion', 'merchant_category_fuel', 'merchant_category_gaming', 'merchant_category_grocery', 'merchant_category_online_services', 'merchant_category_other', 'merchant_category_pharmacy', 'merchant_category_restaurant', 'merchant_category_travel', 'merchant_category_utilities', 'country_AE', 'country_BR']


In [10]:
# Aperçu du dataset encodé
X_encoded.head()

,amount,hour,country_risk,distance_km,tx_last_1h,tx_last_24h,device_change,is_international,merchant_category_atm,merchant_category_electronics,...,country_AE,country_BR,country_CN,country_ES,country_FR,country_MA,country_NG,country_RU,country_UK,country_US
0,11.95,12,0.044,13.91,4,8,0,0,False,False,...,False,False,False,False,False,True,False,False,False,False
1,10.29,11,0.136,13.17,1,4,0,0,False,False,...,False,False,False,False,False,True,False,False,False,False
2,97.04,19,0.075,51.35,0,2,0,0,False,False,...,False,False,False,False,False,True,False,False,False,False
3,16.60,19,0.047,21.52,3,7,0,0,True,False,...,False,False,False,False,False,True,False,False,False,False
4,70.93,7,0.396,47.09,0,2,0,1,False,False,...,False,False,True,False,False,False,False,False,False,False


## Q5) Split des données (70% train / 15% validation / 15% test)

In [12]:
# 1ère division : 70% train | 30% temp (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_encoded, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# 2ème division : 50% val | 50% test (soit 15% chacun du total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Répartition des ensembles :")
print(f"  Train      : {X_train.shape[0]} ({X_train.shape[0]/len(X_encoded)*100:.1f}%)")
print(f"  Validation : {X_val.shape[0]} ({X_val.shape[0]/len(X_encoded)*100:.1f}%)")
print(f"  Test       : {X_test.shape[0]} ({X_test.shape[0]/len(X_encoded)*100:.1f}%)")

print("\nProportion de fraude dans chaque ensemble :")
print(f"  Train      : {y_train.mean()*100:.2f}%")#-y_train contient 1 ou 0   mean() calcule le moyenne par exemple  Moyenne = (0+0+1+0+1)/5=0.4 equivalent a 40%
print(f"  Validation : {y_val.mean()*100:.2f}%")
print(f"  Test       : {y_test.mean()*100:.2f}%")
print("danc ona conserve la proportion du fraude dans les trois ensemble ")

Répartition des ensembles :
  Train      : 3659 (70.0%)
  Validation : 784 (15.0%)
  Test       : 785 (15.0%)

Proportion de fraude dans chaque ensemble :
  Train      : 14.87%
  Validation : 14.80%
  Test       : 14.90%
danc ona conserve la proportion du fraude dans les trois ensemble 


## Q6) Mise à l'échelle (Standardisation) comme ona fait dans le cours

In [13]:
# Initialisation du StandardScaler
scaler = StandardScaler()

# Ajustement UNIQUEMENT sur les données d'entraînement
X_train_sc = scaler.fit_transform(X_train)

# Application de la transformation sur validation et test
X_val_sc  = scaler.transform(X_val)
X_test_sc = scaler.transform(X_test)

print("Standardisation appliquée :")
print(f"  X_train_sc : {X_train_sc.shape}")
print(f"  X_val_sc   : {X_val_sc.shape}")
print(f"  X_test_sc  : {X_test_sc.shape}")

print(f"\nMoyenne sur train (doit ≈ 0) : {X_train_sc.mean():.6f}")
print(f"Écart-type sur train (doit ≈ 1) : {X_train_sc.std():.6f}")

Standardisation appliquée :
  X_train_sc : (3659, 30)
  X_val_sc   : (784, 30)
  X_test_sc  : (785, 30)

Moyenne sur train (doit ≈ 0) : -0.000000
Écart-type sur train (doit ≈ 1) : 1.000000


## Q7) Construction du modèle MLP

In [14]:
# Nombre de variables d'entrée
n_features = X_train_sc.shape[1]
print(f"Nombre de variables d'entrée : {n_features}")

# Construction du modèle Sequential
model = keras.Sequential([
    # Couche d'entrée
    layers.Input(shape=(n_features,)),

    # 1ère couche cachée : 64 neurones, activation ReLU
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.30),
    # 2ème couche cachée : 32 neurones, activation ReLU
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.20)

    # Couche de sortie : 1 neurone, activation Sigmoid (classification binaire)
    layers.Dense(1, activation='sigmoid')
])

# Résumé du modèle
model.summary()

Nombre de variables d'entrée : 30


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,097 (16.00 KB)

 Trainable params: 4,097 (16.00 KB)

 Non-trainable params: 0 (0.00 B)

## Q8) Compilation du modèle

**Question : Pourquoi le recall est-il une métrique importante dans la détection de fraude ?**

Le **recall** (ou sensibilité) mesure la proportion de fraudes réelles correctement détectées par le modèle.  
Dans le contexte de la détection de fraude, **manquer une fraude FN  (faux négatif) est beaucoup plus coûteux** que déclencher une fausse alerte (faux positif). Une fraude non détectée peut entraîner des pertes financières importantes pour la banque et le client.  
Il est donc primordial de maximiser le recall pour s'assurer que le modèle détecte le maximum de transactions frauduleuses, même au prix de quelques fausses alertes supplémentaires.

In [15]:
# Compilation du modèle
model.compile(
    loss='binary_crossentropy',          # Fonction de perte pour classification binaire
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=['accuracy',
             keras.metrics.Recall(name='recall')]
)
#pour clarifie pas plus
print("Modèle compilé avec succès.")
print("  - Loss     : binary_crossentropy")
print("  - Optimizer: Adam (lr=1e-3)")
print("  - Métriques: accuracy, recall")

Modèle compilé avec succès.
  - Loss     : binary_crossentropy
  - Optimizer: Adam (lr=1e-3)
  - Métriques: accuracy, recall


## Q9) Gestion du surapprentissage (Early Stopping)

In [17]:
# Callback Early Stopping
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,                # Stopper si pas d'amélioration pendant 5 epochs
    restore_best_weights=True  # Restaurer les poids de la meilleure epoch
)
#de meme
print("Early Stopping configuré :")
print("  - Monitor : val_loss")
print("  - Patience : 5 epochs")
print("  - restore_best_weights : True")

Early Stopping configuré :
  - Monitor : val_loss
  - Patience : 5 epochs
  - restore_best_weights : True


## Q10) Gestion du déséquilibre des classes

In [18]:
# Q10) class_weight
#   donne plus de poids à la classe minoritaire (fraude)

neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
class_weight = {0: 1.0, 1: (neg / max(pos, 1))}
print("\nclass_weight:", class_weight)

print(f"\nDétail :")
print(f"  Transactions normales (neg) : {neg}")
print(f"  Transactions frauduleuses (pos) : {pos}")
print(f"  Ratio neg/pos : {neg/pos:.2f}")
print(f"  → Les erreurs sur la classe fraude sont {neg/pos:.1f}x plus pénalisées")


class_weight: {0: 1.0, 1: 5.726102941176471}

Détail :
  Transactions normales (neg) : 3115
  Transactions frauduleuses (pos) : 544
  Ratio neg/pos : 5.73
  → Les erreurs sur la classe fraude sont 5.7x plus pénalisées


## Q11) Entraînement du modèle

In [19]:
# Entraînement du modèle
history = model.fit(
    X_train_sc, y_train,
    validation_data=(X_val_sc, y_val),
    epochs=200,
    batch_size=64,
    class_weight=class_weight,
    callbacks=[early_stopping],
    verbose=1
)

print(f"\nEntraînement terminé à l'epoch : {len(history.history['loss'])}")
print(f"Meilleure val_loss obtenue   : {min(history.history['val_loss']):.4f}")

Epoch 1/200
58/58 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.6437 - loss: 1.2524 - recall: 0.4082 - val_accuracy: 0.5191 - val_loss: 0.6951 - val_recall: 0.7500
Epoch 2/200
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6011 - loss: 1.1094 - recall: 0.6203 - val_accuracy: 0.5625 - val_loss: 0.6614 - val_recall: 0.7155
Epoch 3/200
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6356 - loss: 1.0646 - recall: 0.6663 - val_accuracy: 0.5995 - val_loss: 0.6442 - val_recall: 0.7069
Epoch 4/200
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6427 - loss: 1.0526 - recall: 0.6621 - val_accuracy: 0.6135 - val_loss: 0.6315 - val_recall: 0.6810
Epoch 5/200
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6546 - loss: 1.0575 - recall: 0.6624 - val_accuracy: 0.6122 - val_loss: 0.6358 - val_recall: 0.6897
Epoch 6/200
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6532 - loss: 1.0513 - recall: 0.6504 - val_accuracy: 0.6199 - val_loss: 0.6325 - val_recall: 0.6724
Epoch 7/2

## Q12) Analyse des courbes d'apprentissage
pas fait car vous avez dit madam il na pas encore fait dans le cours

## Q13) Évaluation du modèle sur l'ensemble de test

- **Fausses alertes (Faux Positifs)** : transactions normales classifiées comme frauduleuses → valeur `cm[0][1]`
- **Fraudes non détectées (Faux Négatifs)** : transactions frauduleuses classifiées comme normales → valeur `cm[1][0]`

In [20]:
# 13) Évaluation sur TEST
# Distribution des classes dans le test
print("\n--- Distribution des classes dans TEST ---")
print("Normal (0) :", (y_test == 0).sum())
print("Fraude (1) :", (y_test == 1).sum())
test_loss, test_acc, test_recall = model.evaluate(X_test_sc, y_test, verbose=0)

print(f"\nTEST → loss={test_loss:.4f} | acc={test_acc:.4f} | recall={test_recall:.4f}")

# Probabilités + prédictions
y_proba = model.predict(X_test_sc).ravel()
y_pred  = (y_proba >= 0.5).astype(int)

# Matrice de confusion
print("\n--- Confusion Matrix (seuil=0.5) ---")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Métriques
print("\n--- Metrics ---")
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1       :", f1_score(y_test, y_pred, zero_division=0))

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraude'], zero_division=0))


--- Distribution des classes dans TEST ---
Normal (0) : 668
Fraude (1) : 117

TEST → loss=0.5836 | acc=0.6854 | recall=0.6239
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

--- Confusion Matrix (seuil=0.5) ---
[[465 203]
 [ 44  73]]

--- Metrics ---
Precision: 0.2644927536231884
Recall   : 0.6239316239316239
F1       : 0.37150127226463103

--- Classification Report ---
              precision    recall  f1-score   support

      Normal       0.91      0.70      0.79       668
      Fraude       0.26      0.62      0.37       117

    accuracy                           0.69       785
   macro avg       0.59      0.66      0.58       785
weighted avg       0.82      0.69      0.73       785



## Q14) Test pour différents seuils de décision

**Quel seuil est le plus adapté pour un système réel de détection de fraude ?**

Dans un système réel de détection de fraude bancaire :
- Un **seuil bas (ex: 0.2 ou 0.3)** maximise le recall → on détecte plus de fraudes, mais on génère plus de fausses alertes.
- Un **seuil élevé (ex: 0.6)** favorise la precision → moins de fausses alertes, mais on rate plus de fraudes.

**Le seuil de 0.2 ou 0.3 est généralement le plus adapté** pour la détection de fraude car :
1. Le coût d'une fraude manquée (FN) est nettement supérieur au coût d'une fausse alerte (FP).
2. Une fausse alerte entraîne au pire une vérification manuelle, tandis qu'une fraude non détectée cause une perte financière réelle.
3. En pratique, les systèmes bancaires préfèrent **bloquer une transaction légitime** et demander confirmation plutôt que de laisser passer une fraude.

In [23]:
# 14) Tester d'autres seuils
def evaluate_threshold(th):
    yp = (y_proba >= th).astype(int)
    p  = precision_score(y_test, yp, zero_division=0)
    r  = recall_score(y_test, yp, zero_division=0)
    f  = f1_score(y_test, yp, zero_division=0)
    return p, r, f

print("\n--- Seuils alternatifs ---")
results = []
for th in [0.6, 0.5, 0.3, 0.2, 0.1]:
    p, r, f = evaluate_threshold(th)
    results.append({'Seuil': th, 'Precision': p, 'Recall': r, 'F1': f})
    print(f"Seuil={th:.1f} -> Precision={p:.4f} | Recall={r:.4f} | F1={f:.4f}")




--- Seuils alternatifs ---
Seuil=0.6 -> Precision=0.3459 | Recall=0.4701 | F1=0.3986
Seuil=0.5 -> Precision=0.2645 | Recall=0.6239 | F1=0.3715
Seuil=0.3 -> Precision=0.1846 | Recall=0.8376 | F1=0.3025
Seuil=0.2 -> Precision=0.1656 | Recall=0.9060 | F1=0.2801
Seuil=0.1 -> Precision=0.1520 | Recall=0.9744 | F1=0.2630
